# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n\nDescription: {metadata.description}\n\nRecord sets: {len(metadata.record_sets)}")

## 2. Data Overview
Review available record sets, their `@id`, fields, and field `@id`s for use in later analysis.

In [ ]:
# List all record sets in the dataset
for record_set in metadata.record_sets:
    print(f"Record set name: {record_set.name}\n  @id: {record_set.id}\n  Description: {record_set.description or 'None'}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print()

## 3. Data Extraction
Load data from specific record set(s) into DataFrame(s) for analysis. All operations in the remainder of this notebook use the `@id` values from above.

In [ ]:
# Identify available record_set @id's
record_set_ids = [rs.id for rs in metadata.record_sets]
print('Record sets IDs:', record_set_ids)

# For this example, we use the first record set for analysis (update to desired @id as needed).
primary_record_set_id = record_set_ids[0] if record_set_ids else None

dataframes = {}
if primary_record_set_id:
    records = list(dataset.records(record_set=primary_record_set_id))
    df = pd.DataFrame(records)
    dataframes[primary_record_set_id] = df
    print(f"Loaded DataFrame for record set {primary_record_set_id} with columns: ")
    print(df.columns.tolist())
    display(df.head())
else:
    print('No record sets found in this dataset. Please check the schema.')

## 4. Exploratory Data Analysis (EDA)
Apply data processing such as filtering, normalization, and grouping for a numeric field in the main record set. All fields are referenced by their `@id`.

In [ ]:
if primary_record_set_id and not dataframes[primary_record_set_id].empty:
    
    # List all field @id and types for the selected record set
    primary_record_set = [rs for rs in metadata.record_sets if rs.id == primary_record_set_id][0]
    print('Field @id and type for selected record set:')
    for f in primary_record_set.fields:
        print(f"  - {f.name}: {f.id}, type={f.data_type}")
    
    # Attempt to select the first numeric (Float or Integer) field for demonstration
    numeric_field = None
    for f in primary_record_set.fields:
        if f.data_type in ('Float', 'Integer', 'Number'):
            numeric_field = f.id
            break
    if numeric_field and numeric_field in dataframes[primary_record_set_id].columns:
        df = dataframes[primary_record_set_id]
        print(f"\nUsing numeric field '{numeric_field}' for EDA.")
        # Filter to values above the median (or 10 if appropriate)
        if pd.api.types.is_numeric_dtype(df[numeric_field]):
            threshold = max(10, df[numeric_field].median())
            filtered_df = df[df[numeric_field] > threshold]
            print(f"Filtered records with {numeric_field} > {threshold}:")
            display(filtered_df.head())

            # Normalize
            norm_col = f"{numeric_field}_normalized"
            filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
            print(f"Normalized {numeric_field} for filtered records:")
            display(filtered_df[[numeric_field, norm_col]].head())

            # Now group by a categorical/text field if present
            group_field = None
            for f in primary_record_set.fields:
                if f.data_type == 'Text' and f.id != numeric_field and f.id in filtered_df:
                    group_field = f.id
                    break

            if group_field:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
                print(f"\nMean of {numeric_field} grouped by {group_field}:")
                display(grouped_df.head())
            else:
                print('No suitable group field found for grouping.')
        else:
            print(f"Field '{numeric_field}' is not numeric in the loaded data.")
    else:
        print('No numeric field found or present in DataFrame columns.')
else:
    print('DataFrame is empty, skipping EDA.')

## 5. Visualization
Visualize a numeric field distribution and its relationship with a categorical field.

The below cell will create simple plots using matplotlib if numeric and group fields are available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if primary_record_set_id and not dataframes[primary_record_set_id].empty:
    df = dataframes[primary_record_set_id]
    # Reuse previously determined fields
    if 'numeric_field' in locals() and numeric_field in df:
        plt.figure(figsize=(6, 4))
        sns.histplot(df[numeric_field].dropna(), kde=True)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel('Count')
        plt.show()
        
        if 'group_field' in locals() and group_field:
            # Boxplot
            plt.figure(figsize=(8, 4))
            sns.boxplot(data=df[[numeric_field, group_field]].dropna(), x=group_field, y=numeric_field)
            plt.title(f"{numeric_field} by {group_field}")
            plt.show()
    else:
        print('No numeric field found for visualization.')

## 6. Conclusion
This notebook demonstrated how to load, explore, and analyze a [Croissant](https://mlcommons.org/croissant/) dataset using the `mlcroissant` library.

- Dataset title: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells
- Metadata and all exploration steps referenced fields by their Croissant `@id`.
- Main record sets and fields were listed and explored, and elementary EDA was performed on the main numeric field(s).
- You can adapt this notebook for other Croissant datasets by modifying the schema URL and analyzable fields.